# The Full RAG Flow

All the pieces, end to end. The pipeline splits cleanly into **preprocessing** (done ahead of time) and **query time** (per user question).

```
PREPROCESSING (once, ahead of time)
1. Chunk the source document
2. Embed each chunk
3. Store the embeddings in a vector database
─────────────────────────────────────────────
QUERY TIME (per question)
4. Embed the user's query (same model)
5. Find the most similar stored embeddings (cosine similarity)
6. Put the top chunk(s) + the question into a prompt → Claude answers
```

This lesson explains the flow and the similarity math with a tiny, made-up example. The **next** lesson (*Implementing the RAG flow*) wires it up for real with Voyage embeddings and `report.md`.

> Pure-Python illustration — no API calls.

## Steps 1–2: chunk & embed (toy model)

Take two sections:

- **Medical Research** — *"...XDR-47, a 'bug' we have not seen before."*
- **Software Engineering** — *"...studying various infection vectors in our distributed systems."*

Real embeddings have hundreds/thousands of dimensions whose meanings we can't read. To build intuition, pretend we have a **perfect 2-number model** where dimension 0 = "how medical" and dimension 1 = "how software." Note the cross-talk: "bug" gives the medical text some software score, and "infection vectors" gives the software text some medical score.

In [1]:
# Toy 2-D embeddings (dim 0 = medical-ness, dim 1 = software-ness)
medical  = [0.97, 0.34]   # mostly medical, some software ("bug")
software = [0.30, 0.97]   # mostly software, some medical ("infection vectors")

chunks = {
    "Medical Research": medical,
    "Software Engineering": software,
}
chunks

{'Medical Research': [0.97, 0.34], 'Software Engineering': [0.3, 0.97]}

## Normalization (handled for you)

Embedding APIs typically **normalize** vectors to magnitude 1.0, so they sit on the unit circle (in 2-D) or unit hypersphere (in N-D). You don't compute this yourself — but note cosine similarity is **scale-invariant** anyway, so it doesn't change the ranking. Normalizing `medical` and `software` gives roughly `[0.944, 0.331]` and `[0.295, 0.955]`.

In [2]:
import math


def normalize(v):
    mag = math.sqrt(sum(x * x for x in v))
    return [x / mag for x in v]


print("medical  ->", [round(x, 3) for x in normalize(medical)])
print("software ->", [round(x, 3) for x in normalize(software)])

medical  -> [0.944, 0.331]
software -> [0.295, 0.955]


## Steps 4–5: embed the query, then rank by cosine similarity

**Cosine similarity** measures the cosine of the angle between two vectors:

$$\cos(\theta) = \frac{A \cdot B}{\lVert A\rVert \, \lVert B\rVert}$$

- ranges **-1 … 1**; ~1 = very similar, 0 = unrelated (perpendicular), ~-1 = opposite
- **scale-invariant** — only direction matters, not magnitude

The query *"what did the software engineering dept do this year?"* embeds as something like `[0.1, 0.89]` — low medical, high software.

In [3]:
def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def magnitude(v):
    return math.sqrt(sum(x * x for x in v))


def cosine_similarity(a, b):
    return dot(a, b) / (magnitude(a) * magnitude(b))


query = [0.1, 0.89]

ranked = sorted(
    ((name, cosine_similarity(query, vec)) for name, vec in chunks.items()),
    key=lambda pair: pair[1],
    reverse=True,
)

for name, score in ranked:
    print(f"{score:.3f}  {name}")

0.982  Software Engineering
0.434  Medical Research


The **Software Engineering** chunk wins by a wide margin (~0.98 vs ~0.43) — so that's what we retrieve.

> The course quotes **0.983** (software) and **0.398** (medical). Ours match on the winner and are close on the loser; the small gap is just rounding in the made-up vectors. The point holds: semantic search surfaces the software section for a software question, even though the medical section literally contains the word *"bug."* That's the whole advantage over keyword search.

## Cosine distance

Vector-DB docs often report **cosine distance = 1 − cosine similarity**:
- ~0 → very similar
- larger → less similar

Same information, flipped so "smaller is closer," which is convenient for nearest-neighbor search.

In [4]:
for name, score in ranked:
    print(f"distance {1 - score:.3f}  {name}")

distance 0.018  Software Engineering
distance 0.566  Medical Research


## Step 6: build the final prompt

Combine the question with the retrieved chunk and send to Claude:

```
Answer the user's question about the report.

<user_question>
What did the software engineering dept do this year?
</user_question>

<report>
## Section 2: Software Engineering
This division dedicated significant effort to studying various infection vectors in our distributed systems
</report>
```

That's the complete loop: chunk → embed → store → (query) embed → rank by cosine similarity → prompt with the top match. **Step 3 (vector database)** is glossed here — for a handful of chunks you can just keep the vectors in a list and compute similarity in Python, which is exactly what the next lesson does before you'd reach for a real vector DB at scale.

Next: **Implementing the RAG flow** — the real thing with Voyage embeddings over `report.md`.